# `contextual-turn-encoder-base` — entrenamiento del encoder contextual (f2) en M2

Entrena el **encoder contextual de turnos** sobre los embeddings base de Dialog2Flow, en
**M2/MPS**, leyendo las bases vía **memmap** (no entran en RAM). Entrena **dos modos**:

- **autoregresivo (AR)** — causal, *headline* del benchmark (análogo aprendido de Dynamic/EMA);
- **bidireccional (Bidi)** — full-context (techo / inducción de estructura).

Excluye del entrenamiento los **diálogos held-out** del benchmark (semilla 42, `heldout.py`).

> Dialog2Flow es **solo el linaje de datos**; este modelo es nuevo y distinto.

## 1. Setup (rutas + imports)

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# >>> ajustá estas rutas en tu M2 <<<
REPO = os.path.expanduser("~/Documents/GitHub/doctorado-unsl")
PKG = os.path.join(REPO, "packages/contextual-turn-embeddings")
RECIPE = os.path.join(PKG, "training/contextual-turn-encoder-base")
CONFIG_PATH = os.path.join(RECIPE, "config.yaml")

# Datos generados por la notebook 01 (full) o el dialogs-2.0.pkl existente (1m).
CORPUS = "full"                                  # "full" | "1m"
DATA_DIR = os.path.expanduser("~/Documents/GitHub/doctorado-unsl/data/d2f-full")
META_PATH = os.path.join(DATA_DIR, "dialogs-full.pkl")
BASE_EMB_PATH = os.path.join(DATA_DIR, "base_embeddings.npy")
# Metadata de la colección de 1M (para reproducir el held-out con semilla 42):
HELDOUT_META = os.path.expanduser("~/Documents/GitHub/ANN-UNSL/data/dialogs-2.0.pkl")
OUT = os.path.join(PKG, "models")                # gitignored

sys.path.insert(0, RECIPE)                        # heldout.py
import heldout as H
from contextual_turn_embeddings import (Config, ContextualTurnModel, DialogueDataset,
    EmbeddingRetrievalConfig, collate_dialogues, compute_objectives, get_device,
    resolve_losses_for_mode, set_seed)
from contextual_turn_embeddings.train import build_linear_warmup_scheduler
print("device:", get_device("mps"))

device: mps


## 2. Cargar datos (memmap) y excluir held-out

In [2]:
df = pd.read_pickle(META_PATH)
keep = ["dialogue_id", "turn_id", "speaker", "utterance"]
df = df[keep].copy()
df["row_id"] = np.arange(len(df), dtype=np.int64)     # = índice en el memmap

emb = np.load(BASE_EMB_PATH, mmap_mode="r")           # NO entra en RAM
assert len(df) == len(emb), (len(df), len(emb))

# Held-out reproducible (semilla 42) sobre la colección de 1M -> dialogue_ids a excluir.
df_1m = pd.read_pickle(HELDOUT_META)
heldout_ids = H.heldout_dialogue_ids(df_1m)
train_mask, _ = H.split_train_heldout(df, heldout_ids)
df_train = df[train_mask].reset_index(drop=True)      # row_id sigue apuntando al memmap

print(f"corpus {CORPUS}: {len(df)} turnos / {df.dialogue_id.nunique()} diálogos")
print(f"held-out: {len(heldout_ids)} diálogos | train: {len(df_train)} turnos / "
      f"{df_train.dialogue_id.nunique()} diálogos")

corpus full: 1974814 turnos / 196365 diálogos
held-out: 17362 diálogos | train: 1720616 turnos / 179003 diálogos


## 3. Función de entrenamiento (memmap, por modo)

In [3]:
def train_variant(df_train, emb_memmap, base_cfg, mode, num_layers, out_dir,
                  retrieval=False, val_frac=0.02, max_epochs=None):
    set_seed(base_cfg.training.seed)
    device = get_device(base_cfg.training.device)

    # val chica por diálogo (para la curva / mejor checkpoint)
    rng = np.random.default_rng(123)
    dids = pd.unique(df_train["dialogue_id"])
    val_dids = set(rng.choice(dids, size=max(1, int(len(dids) * val_frac)), replace=False))
    is_val = df_train["dialogue_id"].isin(val_dids).to_numpy()
    df_tr, df_va = df_train[~is_val], df_train[is_val]

    cfg = Config.from_dict(base_cfg.to_dict())
    cfg.model.attention_mode = mode
    cfg.model.num_layers = num_layers
    D = int(emb_memmap.shape[1])
    cfg.model.input_dim = cfg.model.hidden_dim = cfg.model.output_dim = D
    if max_epochs is not None:
        cfg.training.epochs = max_epochs

    losses = resolve_losses_for_mode(cfg.losses, mode)
    if retrieval:
        losses.embedding_retrieval = EmbeddingRetrievalConfig(
            enabled=True, target=("masked" if mode == "bidirectional" else "next_turn"))

    mk = lambda d: DialogueDataset(d, emb_memmap, max_turns=cfg.data.max_turns,
                                   num_speakers=cfg.model.num_speakers, lazy=True)
    tr_loader = DataLoader(mk(df_tr), batch_size=cfg.training.batch_size, shuffle=True,
                           num_workers=0, collate_fn=collate_dialogues)
    va_loader = DataLoader(mk(df_va), batch_size=cfg.training.batch_size, shuffle=False,
                           num_workers=0, collate_fn=collate_dialogues)

    model = ContextualTurnModel(cfg.model).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.training.learning_rate,
                            weight_decay=cfg.training.weight_decay)
    total = max(1, len(tr_loader) * cfg.training.epochs)
    sched = build_linear_warmup_scheduler(opt, int(total * cfg.training.warmup_ratio), total)

    def move(b):
        o = dict(b)
        o["embeddings"] = b["embeddings"].to(device)
        o["attention_mask"] = b["attention_mask"].to(device)
        if b.get("speaker_ids") is not None:
            o["speaker_ids"] = b["speaker_ids"].to(device)
        return o

    @torch.no_grad()
    def val():
        model.eval(); set_seed(999); tot = n = 0
        for b in va_loader:
            b = move(b); out = compute_objectives(model, b, losses)
            bs = b["embeddings"].shape[0]; tot += float(out["total"].detach().cpu()) * bs; n += bs
        model.train(); return tot / max(1, n)

    os.makedirs(out_dir, exist_ok=True)
    logf = open(os.path.join(out_dir, "trainlog.jsonl"), "w"); best = float("inf"); t0 = time.time()
    for ep in range(1, cfg.training.epochs + 1):
        model.train(); run = 0.0; te = time.time()
        for i, b in enumerate(tr_loader):
            b = move(b); opt.zero_grad(set_to_none=True)
            loss = compute_objectives(model, b, losses)["total"]; loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.training.gradient_clip_norm)
            opt.step(); sched.step(); run += float(loss.detach().cpu())
            if i % cfg.training.log_interval == 0:
                print(f"[{mode}] ep{ep} {i}/{len(tr_loader)} loss={float(loss.detach().cpu()):.4f}", flush=True)
        rec = {"mode": mode, "epoch": ep, "train_loss": round(run / max(1, len(tr_loader)), 5),
               "val_loss": round(val(), 5), "epoch_sec": round(time.time() - te, 1)}
        print("EPOCH", json.dumps(rec), flush=True); logf.write(json.dumps(rec) + "\n"); logf.flush()
        model.save_pretrained(out_dir, training_args=rec)
        if rec["val_loss"] < best:
            best = rec["val_loss"]; model.save_pretrained(os.path.join(out_dir, "best"), training_args=rec)
    cfg.to_yaml(os.path.join(out_dir, "config.yaml"))
    print(f"[{mode}] DONE best_val={best:.5f} min={(time.time() - t0) / 60:.1f}", flush=True)
    return model

## 4. Entrenar AR + Bidi

In [4]:
base_cfg = Config.from_yaml(CONFIG_PATH)
NAME = "contextual-turn-encoder-base"
NLAYERS = 6 if CORPUS == "full" else 4

# retrieval=True -> contrastivo co-primario (target se resuelve por modo, sin combo leaky).
# El residual-to-base (h_t = LayerNorm(e_t + delta)) viene activado desde config.yaml.
m_ar = train_variant(df_train, emb, base_cfg, "autoregressive", NLAYERS,
                     os.path.join(OUT, f"{NAME}-ar-{CORPUS}"), retrieval=True)
m_bidi = train_variant(df_train, emb, base_cfg, "bidirectional", NLAYERS,
                       os.path.join(OUT, f"{NAME}-bidi-{CORPUS}"), retrieval=True)
print("Checkpoints en:", OUT)

[autoregressive] ep1 0/1371 loss=8.0281
[autoregressive] ep1 100/1371 loss=6.5414
[autoregressive] ep1 200/1371 loss=5.7930
[autoregressive] ep1 300/1371 loss=5.1542
[autoregressive] ep1 400/1371 loss=4.8954
[autoregressive] ep1 500/1371 loss=4.5784
[autoregressive] ep1 600/1371 loss=4.3675
[autoregressive] ep1 700/1371 loss=4.4583
[autoregressive] ep1 800/1371 loss=4.0936
[autoregressive] ep1 900/1371 loss=3.9003
[autoregressive] ep1 1000/1371 loss=3.9668
[autoregressive] ep1 1100/1371 loss=3.7470
[autoregressive] ep1 1200/1371 loss=3.8811
[autoregressive] ep1 1300/1371 loss=3.6951
EPOCH {"mode": "autoregressive", "epoch": 1, "train_loss": 4.68492, "val_loss": 5.10104, "epoch_sec": 1151.5}
[autoregressive] ep2 0/1371 loss=3.7427
[autoregressive] ep2 100/1371 loss=3.7367
[autoregressive] ep2 200/1371 loss=3.5287
[autoregressive] ep2 300/1371 loss=3.7541
[autoregressive] ep2 400/1371 loss=3.5694
[autoregressive] ep2 500/1371 loss=3.6748
[autoregressive] ep2 600/1371 loss=3.6228
[autoreg

## 5. Empaquetar para el Release (interno)

Los pesos no van a git (`models/` está gitignored). Se persistirán como **asset de un GitHub Release**
del repo `doctorado-unsl` y se documentará el tag en el `README.md` (model card). Ej.:

```bash
cd ~/Documents/GitHub/doctorado-unsl
# comprimir cada checkpoint y subirlo como asset
tar czf cte-base-ar-full.tar.gz -C packages/contextual-turn-embeddings/models contextual-turn-encoder-base-ar-full
gh release create cte-base-v0 --title "contextual-turn-encoder-base v0 (interno)" --notes "AR/Bidi full+1m" --prerelease
gh release upload cte-base-v0 cte-base-ar-full.tar.gz
```

In [5]:
names = sorted(os.listdir(OUT)) if os.path.isdir(OUT) else []
for name in names:
    p = os.path.join(OUT, name)
    if os.path.isdir(p):
        print(name, "->", [f for f in os.listdir(p) if f.endswith((".safetensors", ".json"))][:4])

contextual-turn-encoder-base-ar-full -> ['model.safetensors', 'config.json', 'training_args.json']
contextual-turn-encoder-base-bidi-full -> ['model.safetensors', 'config.json', 'training_args.json']
